Overview 
- Show a simple 3rd order polynomial function (done)
- Fit a polynomial function to the data after sampling (done)
- Manually fit 3 relu-neurons to the data using interactive sliders (done)
- Learn forward and backward propagation and implement it in code for one step of gradient descent for a three neuron network
- Build a complete neural network with one hidden layer and many neurons and train it to learn the function
- Compare with polynomial regression and see how it performs when we add noise to the data.

# Building a neural network from scratch

In this module, you will build a neural network which will learn a very simple function, a polynomial function of the sort: 
y = A + B * x + C * x^2 + D * x^3 

You can choose the value of coefficients for this exercise. Later on we will see what happens when we add noise. 

But first, let us plot and see your function! 

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(0)
A = 1 
B = 2
C = -0.5
D = 0.04
x_true = np.linspace(0, 10, 1000).round(2)
y_true = (A + B * x_true + C * x_true**2 + D * x_true**3).round(2)

plt.plot(x_true, y_true, color="black")
plt.xlabel('X')
plt.ylabel('Y')

# Learning from data
The function that you plot above is ofcourse arbitrary, since you chose the coefficients of the polynomial. In fact the choice of a polynomial is also arbitrary (we chose it because it is easy to explain). But in reality this could have a certain meaning; a 1-D function could be a signal from a sensor, for instance a temperature sensor, or it could show how price of a house changes with time, etc. 
(Insert example of a 1-D signal relevant to biology, which is simple like a polynomial)

In reality, you will get only discrete data points from a sensor and you need to build a model from this data. The process of learning the function from a discrete set of points is called Machine Learning. Below, you can see a set of sampled points from the function that you generated above. The sampled points are in red, and the dashed line indicates the true function. 

In [ ]:
sample_size = 15
sampling_indices = np.random.choice(len(x_true), size=sample_size, replace=False)
x_sample = x_true[sampling_indices]
y_sample = y_true[sampling_indices] 

plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--')
plt.scatter(x_sample, y_sample, color="red", label="Sampled Data")
plt.legend()

# show sampled points as a table 

data = {'X': x_sample, 'Y': y_sample}
df = pd.DataFrame(data)
print(df.head())

A direct method is ofcourse to use a polynomial function. We will use that method to fit the function first. Then, we will build a neural network from scratch and then fit those same datapoints. We will then discuss what the difference is between fitting a polynomial versus a neural network is. 

In [ ]:
# Fitting a polynomial of degree 3 to the data
coefficients = np.polyfit(x_sample, y_sample, deg=3)
fitted_polynomial = np.poly1d(coefficients)
y_fitted = fitted_polynomial(x_true)
plt.plot(x_true, y_true, color="black", label="True Function", linestyle='--', linewidth=4)
plt.plot(x_true, y_fitted, color="blue", label="Fitted Polynomial")
plt.scatter(x_sample, y_sample, color="red", label="Sampled Data")
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()


# Learning through neurons

We will see in this module how the same function can be learnt through a different framework, one loosely inspired by the human brain. A neuron is simply a cell which takes some input and transforms into an output, and the comparison to biological neuron basically ends there. 

For biological neurons, the transformation of the input is generally described as a binary output (either it fires or not). This can be modelled as a sigmoid function, where the output slowly changes between 0 and 1 through an exponential change. 

A digital neuron can ofcourse be modelled by any mathematical function. Normally, this function is a non-linear function. The simplest non-linear function that is typically used is one which just produces a null output for all negative inputs, and returns the same input if it is positive. It is called a rectified linear function (ReLU). Run the code cell below to see the shape of the two functions 

TODO: Add an interactive cell here to show how neurons modify the input given an activation function

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
def activation_function(z, function_type="sigmoid"):
    if function_type == "sigmoid":
        return 1 / (1 + np.exp(-z))
    elif function_type == "relu":
        return np.maximum(0, z)
    elif function_type == "tanh":
        return np.tanh(z)
    elif function_type == "linear":
        return z
    else:
        raise ValueError("Unsupported activation function.")


# plot sigmoid and relu

z = np.linspace(-10, 10, 400)
sigmoid_values = activation_function(z, function_type="sigmoid")
relu_values = activation_function(z, function_type="relu")
tanh_values = activation_function(z, function_type="tanh")
fig, ax = plt.subplots(1, 3, figsize=(6, 2))
ax[0].plot(z, sigmoid_values)
ax[0].set_title("Sigmoid")
ax[1].plot(z, relu_values)
ax[1].set_title("ReLU")
ax[2].plot(z, tanh_values)
ax[2].set_title("Tanh")

plt.tight_layout()



Learning through a neuron network, simply amounts to finding a combination of activation functions which best approximates the dataset that you have provided. In contrast to fitting a polynomial, where you know the type of function that you are approximating, here the prior knowledge required is much less. As you will see, _any_ function can be approximated by a neural network, given enough neurons and layers. 

First, check out what parameters you can change for each activation function type. Let us stick to ReLu for now. 

In [ ]:
# Add an interact to visualise effect of parameters on the neuron output
# Use ReLU, PReLU, sigmoid, and tanh as activation functions
import ipywidgets as widgets
def update_neuron_output(slope, bias):
    input = np.array([1])  # Example input
    # Plot the output
    x_global = np.linspace(-10, 10, 400)
    z = slope * x_global + bias
    y_global = activation_function(z, function_type="relu")
    plt.figure(figsize=(3, 3))
    plt.plot(x_global, y_global, label=f"Output")
    plt.plot(x_global, z, color="gray", linestyle='--', label=f"Linear function")
    plt.xlabel('Input (x)')
    plt.ylabel('Output (y)')
    plt.xlim(0, 10)
    plt.xticks(np.arange(-4, 11, 4))
    plt.yticks(np.arange(-5, 11, 5))
    plt.ylim(-5, 10)
    plt.legend()
    plt.title(f"Slope={slope:.2f}, Bias={bias:.2f}")
    
slope_slider = widgets.FloatSlider(min=0, max=5, step=0.1, value=1, description='Slope')
bias_slider = widgets.FloatSlider(min=-5, max=5, step=0.1, value=0, description='Bias')
widgets.interact(update_neuron_output, slope=slope_slider, bias=bias_slider)



See how the slope and the bias both affect the point at which the function starts to produce a non-zero output. Let us call this as the "activation point" (which is not exactly a technical term, but it is a useful one for our discussion).

Try the next cell to change the ReLU function using slope and activation point. 

In [ ]:
# Add an interact to visualise effect of parameters on the neuron output
# Use ReLU, PReLU, sigmoid, and tanh as activation functions
import ipywidgets as widgets
def update_neuron_output_using_activation_point(slope, activation_point):
    input = np.array([1])  # Example input
    # Plot the output
    x_global = np.linspace(-10, 10, 400)
    # calculate bias from activation point
    bias = -slope * activation_point
    z = slope * x_global + bias
    y_global = activation_function(z, function_type="relu")
    plt.figure(figsize=(3, 3))
    plt.plot(x_global, y_global, label=f"Output")
    plt.plot(x_global, z, color="gray", linestyle='--', label=f"Linear function")
    plt.xlabel('Input (x)')
    plt.ylabel('Output (y)')
    plt.xlim(0, 10)
    plt.xticks(np.arange(-4, 11, 4))
    plt.ylim(-5, 10)
    # vertical line at activation point until it hits the bias value
    plt.scatter(0, bias)
    plt.plot([-4, 0], [bias, bias], color="red", linestyle='--')
    # add bias value as y tick
    plt.yticks([-5.0, 0.0, 5.0, 10.0, bias])
    # round off 1 decimal for ytick label
    plt.yticks([-5.0, 0.0, 5.0, 10.0, bias], [f"{-5:.1f}", f"{0:.1f}", f"{5:.1f}", f"{10:.1f}", f"{bias:.1f}"])
    


    plt.legend()
    plt.title(f"Slope={slope:.2f}, Bias={bias:.2f}")
    plt.tight_layout()
    
slope_slider = widgets.FloatSlider(min=0, max=5, step=0.1, value=1, description='Slope')
activation_point_slider = widgets.FloatSlider(min=0, max=8, step=0.1, value=0, description='Activation point')
widgets.interact(update_neuron_output_using_activation_point, slope=slope_slider, activation_point=activation_point_slider)



Now, combine three ReLU neurons together to see how they can approximate a more complex function. Use activation point, slope and weights to change the shape of the function.

In [ ]:
def update_layer_of_neurons_using_slope_activation_point(slope_1, slope_2, slope_3, activation_point_1, activation_point_2, activation_point_3, weight_1, weight_2, weight_3, output_bias):
    # calculate bias from activation points for each neuron
    bias_1 = -slope_1 * activation_point_1
    bias_2 = -slope_2 * activation_point_2
    bias_3 = -slope_3 * activation_point_3
    # combined output
    x_global = np.linspace(-10, 10, 1000)
    z_1_global = slope_1 * x_global + bias_1
    z_2_global = slope_2 * x_global + bias_2
    z_3_global = slope_3 * x_global + bias_3
    y_1_global = activation_function(z_1_global, function_type="relu")
    y_2_global = activation_function(z_2_global, function_type="relu")
    y_3_global = activation_function(z_3_global, function_type="relu")

    combined_output = weight_1 * y_1_global + weight_2 * y_2_global + weight_3 * y_3_global + output_bias
    x_global_true = np.linspace(0, 10, 1000)
    y_1_global_true = activation_function(slope_1 * x_global_true + bias_1, function_type="relu")
    y_2_global_true = activation_function(slope_2 * x_global_true + bias_2, function_type="relu")
    y_3_global_true = activation_function(slope_3 * x_global_true + bias_3, function_type="relu")
    combined_output_true = weight_1 * y_1_global_true + weight_2 * y_2_global_true + weight_3 * y_3_global_true + output_bias

    fig, ax = plt.subplots(1, 4, figsize=(12, 3))
    ax[0].plot(x_global, y_1_global, label=f"Neuron 1 Output")
    ax[0].set_title(f"Neuron 1\nSlope={slope_1:.2f}, Bias={bias_1:.2f}")
    ax[0].set_xlabel('Input (x)')
    ax[0].set_ylabel('Output (y)')
    ax[0].legend()
    ax[0].set_xlim(-4, 10)
    ax[0].set_xticks(np.arange(-4, 11, 4))
    ax[0].set_ylim(-5, 10)
    ax[0].set_yticks(np.arange(-5, 11, 5))
    ax[1].plot(x_global, y_2_global, label=f"Neuron 2 Output")
    ax[1].set_title(f"Neuron 2\nSlope={slope_2:.2f}, Bias={bias_2:.2f}")
    ax[1].set_xlabel('Input (x)')
    # hide y axis for ax[1]
    ax[1].set_ylim(-5, 10)
    ax[1].set_yticks([])
    ax[1].legend()
    ax[1].set_xlim(0, 10)
    ax[1].set_xticks(np.arange(-4, 11, 4))
    ax[2].plot(x_global, y_3_global, label=f"Neuron 3 Output")
    ax[2].set_title(f"Neuron 3\nSlope={slope_3:.2f}, Bias={bias_3:.2f}")
    ax[2].set_xlabel('Input (x)')
    # hide y axis for ax[2]
    ax[2].set_ylim(-5, 10)
    ax[2].set_yticks([])
    ax[2].legend()
    ax[2].set_xlim(0, 10)
    ax[2].set_xticks(np.arange(-4, 11, 4))
    ax[3].plot(x_global, combined_output, label=f"Combined Output")
    ax[3].plot(x_true, y_true, color="black", label="True Function", linestyle='--', linewidth=2)
    ax[3].set_title(f"Combined Output")
    ax[3].set_xlabel('Input (x)')
    # hide y axis for ax[3]
    ax[3].set_ylim(-5, 10)
    ax[3].set_yticks([])
    ax[3].legend()
    ax[3].set_xlim(0, 10)
    ax[3].set_xticks(np.arange(0, 11, 2))

    # compute mean squared error between combined output and true function at the sampled points
    
    mean_squared_error = np.mean((combined_output_true - y_true)**2)
    plt.suptitle(f"Mean Squared Error: {mean_squared_error:.2f}")
    plt.tight_layout()

In [ ]:
slopes = [widgets.FloatSlider(min=-5, max=5, step=0.1, value=0, description='Slope 1'),
          widgets.FloatSlider(min=-5, max=5, step=0.1, value=0, description='Slope 2'),
          widgets.FloatSlider(min=-5, max=5, step=0.1, value=0, description='Slope 3')]
activation_points = [widgets.FloatSlider(min=0, max=8, step=0.1, value=0, description='Activation point 1'),
                     widgets.FloatSlider(min=0, max=8, step=0.1, value=0, description='Activation point 2'),
                     widgets.FloatSlider(min=0, max=8, step=0.1, value=0, description='Activation point 3')]
weights = [widgets.FloatSlider(min=-5, max=5, step=0.1, value=1, description='Weight 1'),
              widgets.FloatSlider(min=-5, max=5, step=0.1, value=1, description='Weight 2'),
              widgets.FloatSlider(min=-5, max=5, step=0.1, value=1, description='Weight 3')]

output_bias = widgets.FloatSlider(min=-5, max=5, step=0.1, value=0, description='Output Bias')
widgets.interact(update_layer_of_neurons_using_slope_activation_point, \
                 slope_1=slopes[0], slope_2=slopes[1], slope_3=slopes[2], \
                 activation_point_1=activation_points[0], activation_point_2=activation_points[1], activation_point_3=activation_points[2], \
                 weight_1=weights[0], weight_2=weights[1], weight_3=weights[2], output_bias=output_bias)


Could you get a good fit? Could you get the error to be less than 1? What about 0.1? or even 0.01 or zero?

## Think! 
- Why do you think three neurons were sufficient for this curve? 
- What would happen if you had only two neurons? What if you have a million neurons?


# Artificial Neural Networks

In biological neurons, the connection strengths between the cells are modulated by complex biochemical processes. In artificial neurons, the connection strength is simply a number, which we call a weight. The weight can be positive or negative, and it determines how much influence the input has on the output of the neuron. Each neuron performs a linear combination of the inputs with the weights and then applies the non-linear activation function before sending the output to the next layer of neurons. Mathematically, the output of a single neuron can be expressed as:

\begin{equation}
y = f\left(\sum_{i=1}^{n} w_i x_i + b\right)
\end{equation}

Where:
- $x_i$ are the inputs to the neuron
- $w_i$ are the weights associated with each input, similar to the slope that you just played with
- $b$ is the bias term, which shifts the activation point of the function along with the weights
- $f$ is the activation function (e.g., ReLU, sigmoid)
- $y$ is the output of the neuron

To build this in Python, we will make use of Class objects. In the next code cell, you will see a simply definition of a Neuron class. In the initialisation function, you define a certain parameters. Then, you will different functions defined within a given class called methods. 


In [ ]:
class Neuron:
    def __init__(self, weights=None, bias=None, input_vector=None, activation_function_type="relu"):
        self.weights = weights if weights is not None else np.random.normal(0, 1) # initialize weights randomly if not provided
        self.bias = bias if bias is not None else np.random.normal(0, 1)
        self.input_vector = input_vector if input_vector is not None else np.random.normal(0, 1, size=(5,)) # initialize input vector randomly if not provided
        self.activation_function_type = activation_function_type



Nice. In the next cell, you will see a new method called "forward". This method takes an input and produces an output by applying the weights, bias and activation function. We will inherit the Neuron class that was already defined. So you would have to run the previous code cell before you can run the next one.


In [ ]:
class Neuron(Neuron):
    def forward(self):
        z = np.dot(self.weights, self.input_vector) + self.bias
        output = activation_function(z, function_type=self.activation_function_type)
        return output

In [ ]:
n1 = Neuron(
    input_vector = [1, 2, 3], 
    weights = [0.5, 3.2, -0.5], 
    bias = 1, 
    activation_function_type = 'relu'
)

output = n1.forward()

print(f"Output of the neuron: {output:.2f} for input vector {n1.input_vector}, weights {n1.weights}, and bias {n1.bias}")

Next, we will define a layer of neurons. Each layer has a certain number of neurons and commonly they all share the same activation function. The output of one layer is the input to the next layer. The first layer is called the input layer, and the last layer is called the output layer. The layers in between are called hidden layers.


In [ ]:
class Layer:
    def __init__(self):
        self.neurons = []
    def add_neuron(self, neuron):
        # to add single neuron to the layer
        self.neurons.append(neuron)
    def add_neurons(self, neurons):
        # to add multiple neurons to the layer
        self.neurons.extend(neurons)
    

Now, we will add a forward method for the layer to combine the outputs of all the neurons in the layer. The size of the output of the layer will depend on the number of neurons in that layer. 


In [ ]:
class Layer(Layer):
    def __init__(self):
        super().__init__() # call the init method of the parent class to initialize the neurons list
        self.outputs = [] # adds a new attribute to store the outputs of the neurons in the layer
    
    def forward(self):
        outputs = []
        for neuron in self.neurons:
            output = neuron.forward()
            outputs.append(output)
        
        outputs = np.array(outputs)
        #outputs = outputs.squeeze() 
        self.outputs = outputs
        return outputs

In [ ]:
# def layer(**neurons):
#     layer_output = {}
#     for neuron_name, neuron_params in neurons.items():
#         input = neuron_params['x_input']
#         weight = neuron_params['weight']
#         bias = neuron_params['bias']
#         activation_function_type = neuron_params['activation_function_type']
#         output = neuron(input, weight, bias, activation_function_type=activation_function_type)
#         layer_output[neuron_name] = output
#     return layer_output

Let us build an input layer with 1 neuron. Use the next code cell to instantiate a layer with 1 neuron. Use the inputs from sampled data points from before to initialize the input layer.

In [ ]:
# Start with the first input point from the sampled data. 
input_vector = np.array([x_sample[0]])
output_vector = np.array([y_sample[0]])
print(f"The input is {input_vector} and the expected output is {output_vector}")

In [ ]:
input_layer = Layer()
input_layer.add_neuron(
    Neuron(
        input_vector = input_vector, 
        weights = np.array([1]),
        bias = 0, 
        activation_function_type = 'linear'
    )
)



The layers between the input and output layer are called hidden layers. We will use a single hidden layer which has three neurons. Choose your own weights and bias vectors for this hidden layer. Take care to match the dimensions of these parameters! 

In [ ]:
hidden_layer_1 = Layer()
hidden_layer_1.add_neurons([
    Neuron(
        input_vector = input_layer.forward(), 
        weights = np.array([0.5]),
        bias = np.array([0.1]),
        activation_function_type = 'relu'
    ),
    Neuron(
        input_vector = input_layer.forward(), 
        weights = np.array([-0.5]),
        bias = np.array([-0.2]),
        activation_function_type = 'relu'
    ),
    Neuron(
        input_vector = input_layer.forward(), 
        weights = np.array([0.3]),
        bias = np.array([0]),
        activation_function_type = 'relu'
    )
])

    

In [ ]:

hidden_layer_output = hidden_layer_1.forward()

print("Output of the layer:", hidden_layer_output)

Now, we need to combine these neurons into a single output neuron. This will be our final layer. Here, we have one neuron which takes three inputs, which are the outputs of the three neurons in the previous layer. The neuron in this layer will normally have linear activation function, which means that it will simply sum the inputs with the weights and bias without applying any non-linearity.


In [ ]:
final_layer = Layer()
final_layer.add_neuron(
    Neuron(
        input_vector = hidden_layer_1.forward(),
        weights = np.array([0.2, -0.8, 0]), 
        bias = np.array([0.1]),
        activation_function_type = 'linear'
    )
)

final_output = final_layer.forward()
print(f"Final output of the network: {final_output} for input vector {input_vector} and expected output {output_vector}")

Is there a lot of error? Well, that is life... We will now start learning how to correct our mistakes! As in life, the first step is to acknowledge the error. In deep learning, we do it with something called as a "loss function". It tells you how far your predictions are from the expected outcome. 

In case of regression problems (like the one we are working on), a common loss function is the Mean Squared Error (MSE), which is calculated as:
\begin{equation}
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
\end{equation}
Where:
- $y_i$ are the true values
- $\hat{y}_i$ are the predicted values from the neural network
- $n$ is the number of data points in the output
There are other loss functions as well. Try to implement the MSE loss function and calculate the loss for your current model.


In [ ]:
# Compute the loss 
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

In [ ]:
loss_value = mean_squared_error(output_vector, final_output)
print("Loss value:", loss_value)

It's great that you now know exactly how wrong you were! Now, time to correct it. 

# Backpropagation

Here, we calculate the gradient of the loss function with respect to each weights and bias in the neural network. This might seem complicated because there are many parameters. However, we will how the chain rule of calculus comes in handy. 

But first, we will start with a simpler problem. We will calculate the derivative of the loss function with respect to the final output of the network. This is the first step in backpropagation. First, recap the equation for the MSE for a single data point:


\begin{equation}
MSE = (y_i - \hat{y}_i)^2
\end{equation}


\begin{equation}
\frac{\partial MSE}{\partial \hat{y}_{final}} = \delta_{out} = -2(y_i - \hat{y}_i) 
\end{equation}

The term $\delta_{out}$ is called the output error signal, and it tells us how much the final output of the network is contributing to the error.

In [ ]:
def gradient_mse(y_true, y_pred):
    return 2 * (y_pred - y_true) / len(y_true)

output_error_signal = gradient_mse(output_vector, final_output)

print("Output error signal:", output_error_signal)


In [ ]:
# Show a plot of MSE for different predicted values around the true value 

window_size = 40
y_pred_around_true = np.linspace(output_vector[0] - window_size, output_vector[0] + window_size, 100)
mse_values = [mean_squared_error(output_vector, y_pred) for y_pred in y_pred_around_true]
plt.figure(figsize=(5 ,3))
plt.plot(y_pred_around_true, mse_values)
plt.xlabel('Final Layer Output (Predicted Value)')
plt.ylabel('Mean Squared Error')
plt.scatter(final_output, loss_value, color='red', label='Current Prediction')
# add a tangent line at the current prediction point with the slope equal to the gradient value
tangent_slope = output_error_signal.squeeze()
tangent = tangent_slope * y_pred_around_true + (loss_value - tangent_slope * final_output)
plt.plot(y_pred_around_true, tangent.squeeze(), color='orange', 
         label=r'$\delta_{out}$' + '={:.2f}'.format(tangent_slope))
plt.ylim(-200, 1000)
plt.legend();



Let us add the gradient calculation as part of the Neuron class as methods. We can create a "forward_gradient" as a new initialisation parameter and store it for later use. Then we will add a new method called "backward" which will calculate teh gradient of the output of that neuron with respect to its weights and bias. 


In [ ]:
class Neuron(Neuron):
    def __init__(self, weights=None, bias=None, input_vector=None, activation_function_type="relu"):
        super().__init__(weights, bias, input_vector, activation_function_type)
        self.forward_gradient = None # to store the gradient of the neuron it connects to in the forward pass
        self.backward_gradient = None # to help the gradient of the neuron it connects to in the backward pass
    def relu_derivative(self, z):
        return np.where(z > 0, 1, 0)
    
    def backward(self):
        # compute the gradient of the output of this neuron with respect to its weights and bias using the chain rule
        all_weights = np.array(self.weights)
        all_inputs = np.array(self.input_vector)
        
        dy_dw = []
        dyi_dy = []
        for i, x_input in enumerate(all_inputs):
            z = np.dot(all_weights, all_inputs) + self.bias
            if self.activation_function_type == 'relu':
                dy_dz = self.relu_derivative(z)
            elif self.activation_function_type == 'linear':
                dy_dz = 1
            else:
                raise ValueError("Unsupported activation function for backward pass.")
            
            dz_dw = x_input
            dz_dx = all_weights[0][i]
            dy_dw.append(dy_dz * dz_dw)
            dyi_dy.append(dy_dz * dz_dx) # backward gradient

        dy_db = self.relu_derivative(z) * 1

        dL_dw = self.forward_gradient * np.array(dy_dw)
        dL_db = self.forward_gradient * dy_db
        dL_dy = self.forward_gradient * np.array(dyi_dy)

        self.gradient_wrt_weights = np.array(dL_dw) 
        self.gradient_wrt_bias = np.array(dL_db)
        self.backward_gradient = np.array(dL_dy)


In [ ]:
# Update the neurons in all layers to take the latest class definition
for layer in [input_layer, hidden_layer_1, final_layer]:
    for neuron in layer.neurons:
        neuron.__class__ = Neuron


In [ ]:
final_layer.neurons[0].forward_gradient = [output_error_signal]
final_layer.neurons[0].backward()

The information that updates all the parameters of the neural network starts from that slope. We use that slope to find the gradients of the loss function with respect to the weights and biases of the network. 

The next set of parameters concerns the output neuron. The forward pass of the output neuron is 

\begin{equation}
\hat{y}_{final} = w_1 y_1 + w_2 y_2 + w_3 y_3 + b
\end{equation}

Where $y_1$, $y_2$ and $y_3$ are the outputs of the three neurons in the previous layer. Note that the output of the final neuron has a linear activation function. 

\begin{equation}
\frac{\partial \hat{y}_{final}}{\partial w_1} = y_1 
\end{equation}

\begin{equation}
\frac{\partial \hat{y}_{final}}{\partial w_2} = y_2
\end{equation}

\begin{equation}
\frac{\partial \hat{y}_{final}}{\partial w_3} = y_3
\end{equation}

\begin{equation}
\frac{\partial \hat{y}_{final}}{\partial b} = 1
\end{equation}
Now, we can use the chain rule to calculate the gradient of the loss function with respect to the weights and bias of the output neuron. For example, for weight $w_1$, we have:



\begin{equation}
\begin{aligned}
\frac{\partial MSE}{\partial w_1} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_1} = \delta_{out} \cdot y_1 \\
\frac{\partial MSE}{\partial w_2} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_2} = \delta_{out} \cdot y_2 \\
\frac{\partial MSE}{\partial w_3} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial w_3} = \delta_{out} \cdot y_3 \\
\frac{\partial MSE}{\partial b} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial b} = \delta_{out}
\end{aligned}
\end{equation}

Now, write the code to calculate the gradients for the weights and bias of the output neuron. 







In [ ]:
print("Gradient with respect to weights:", final_layer.neurons[0].gradient_wrt_weights)
print("Gradient with respect to bias:", final_layer.neurons[0].gradient_wrt_bias)

Now, let us update the weights and bias of only the output neuron using the gradients that we just calculated. We will use a simple update rule called gradient descent, which updates the parameters in the direction of the negative gradient. The update rule is given by:

\begin{equation}
\begin{aligned}
w_i &\leftarrow w_i - \eta \cdot \frac{\partial MSE}{\partial w_i} \\
&\text{for i = 1, 2, 3} \\
b &\leftarrow b - \eta \cdot \frac{\partial MSE}{\partial b}
\end{aligned}
\end{equation}

Where $\eta$ is the learning rate, which determines how much we update the parameters in each iteration. In the next code cell, we will add a new method to the Neuron class called "udpate" which will take the learning rate, and the derivative of the loss function with respect to its weights and bias as input and update them. 



In [ ]:
class Neuron(Neuron):
    def update(self, learning_rate=0.01):
        # update weights and bias based on the loss gradient and learning rate 
        self.weights = np.array([(self.weights - learning_rate * self.gradient_wrt_weights.T).squeeze()])
        self.bias = np.array([(self.bias - learning_rate * self.gradient_wrt_bias).squeeze()])


In [ ]:
# Upgrade all layer and neuron instances to the latest class definitions
for layer in [input_layer, hidden_layer_1, final_layer]:
    for neuron in layer.neurons:
        neuron.__class__ = Neuron

Now, update the weights and bias of the output neuron using the gradients that we calculated. After updating, calculate the new loss and see if it has decreased.

In [ ]:
learning_rate = 0.01

In [ ]:
print(f"Before update - Weights: {final_layer.neurons[0].weights}, Bias: {final_layer.neurons[0].bias}")

In [ ]:
final_layer.neurons[0].gradient_wrt_bias

In [ ]:
final_layer.neurons[0].update(learning_rate=learning_rate)
print(f"After update - Weights: {final_layer.neurons[0].weights}, Bias: {final_layer.neurons[0].bias}")

In [ ]:
# Compute new prediction with updated weights and bias and show how 
final_output_after_one_update = final_layer.forward()

updated_loss_value = mean_squared_error(output_vector, final_output_after_one_update)
plt.figure(figsize=(5 ,3))
plt.plot(y_pred_around_true, mse_values)
plt.xlabel('Final Layer Output (Predicted Value)')
plt.ylabel('Mean Squared Error')
plt.scatter(final_output, loss_value, color='red', label='Previous Prediction')
plt.scatter(final_output_after_one_update, updated_loss_value, color='green', label='Updated Prediction')
plt.legend();


Did the loss decrease after the update? If not, try to adjust the learning rate and see if it helps.

What happens if you keep on increasing the learning rate? 

Now let us also calculate the gradients for the weights and biases of the three neurons in the previous layer. The forward pass for each of these neurons is given by:
\begin{equation}
\begin{aligned}
y_1 &= f(w_{1-1} x + b_{1-1}) = f(z_1) \\
y_2 &= f(w_{2-1} x + b_{2-1}) = f(z_2) \\
y_3 &= f(w_{3-1} x + b_{3-1}) = f(z_3)
\end{aligned}
\end{equation}

Where $f$ is the activation function (ReLU in our case). To make the text simpler, let us denote the input to the activation function for each neuron as $z_i = w_{i-1} x + b_{i-1}$, where $i$ is the index of the neuron (1, 2, or 3). We will also show the equation for the first neuron only. 

Derivative of the $y_1$ with respect to its parameters is given by:
\begin{equation}
\begin{aligned}
\frac{\partial y_1}{\partial w_{1-1}} &= \frac{\partial y_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_{1-1}} \\
\frac{\partial y_1}{\partial b_{1-1}} &= \frac{\partial y_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial b_{1-1}} 
\end{aligned}
\end{equation}

Now, we need to calculate the derivative of the activation function with respect to its input $z_1$. The derivative of the activation function can be represented by the symbol $f'$. For ReLU, the derivative is given by:
For ReLU, the derivative is given by:

\begin{equation}
f'_1 = \frac{\partial y_1}{\partial z_1} = 
\begin{cases}
1 & \text{if } z_1 > 0 \\
0 & \text{if } z_1 \leq 0
\end{cases}
\end{equation}

This gives, 
\begin{equation}
\begin{aligned}
\frac{\partial y_1}{\partial w_{1-1}} &= \frac{\partial y_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_{1-1}} = f'_1 \cdot x \\
\frac{\partial y_1}{\partial b_{1-1}} &= \frac{\partial y_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial b_{1-1}} = f'_1 \cdot 1
\end{aligned}
\end{equation}

What is the derivative of the output value $\hat{y}_{final}$ with respect to the output of the first neuron $y_1$?
\begin{equation}
\frac{\partial \hat{y}_{final}}{\partial y_1} = w_1
\end{equation}
Now, we can use the chain rule to calculate the gradient of the loss function with respect to the weights and bias of the first neuron. For example, for weight $w_{1-1}$, we have:

\begin{equation}
\begin{aligned}
\frac{\partial MSE}{\partial w_{1-1}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial y_1} \cdot \frac{\partial y_1}{\partial w_{1-1}} \\
&= \delta_{out} \cdot w_1 \cdot f'_1 \cdot x
\end{aligned}
\end{equation}
and for bias $b_{1-1}$, we have:
\begin{equation}
\begin{aligned}
\frac{\partial MSE}{\partial b_{1-1}} &= \frac{\partial MSE}{\partial \hat{y}_{final}} \cdot \frac{\partial \hat{y}_{final}}{\partial y_1} \cdot \frac{\partial y_1}{\partial b_{1-1}} \\
&= \delta_{out} \cdot w_1 \cdot f'_1 
\end{aligned}
\end{equation}
Now, write the code to calculate the gradients for the weights and bias of the first neuron.


In [ ]:
final_layer.neurons[0].backward_gradient

In [ ]:
# Propagate the error back to the hidden layer neurons and update their weights and bias as well
# Compute the error signal for each neuron in the hidden layer using the chain rule
hidden_layer_1.neurons[0].forward_gradient = final_layer.neurons[0].backward_gradient[0][0]
hidden_layer_1.neurons[1].forward_gradient = final_layer.neurons[0].backward_gradient[0][1]
hidden_layer_1.neurons[2].forward_gradient = final_layer.neurons[0].backward_gradient[0][2]
# Update the weights and bias of each neuron in the hidden layer
for neuron in hidden_layer_1.neurons:
    neuron.backward()
    neuron.update(learning_rate=learning_rate)
# Compute the new output of the network after the update
final_output_after_hidden_update = final_layer.forward()
updated_loss_value_after_hidden_update = mean_squared_error(output_vector, final_output_after_hidden_update)


In [ ]:
def relu_derivative(z):
    return np.where(z > 0, 1, 0)

In [ ]:
final_layer.neurons[0].weights

In [ ]:
# Compute parameters of the first neuron, using the formula above
dMSE_dW11 = output_error_signal * final_layer.neurons[0].weights[0] * relu_derivative(hidden_layer_1.outputs[0]) * input_layer.forward()[0]
dMSE_dB1 = output_error_signal * final_layer.neurons[0].weights[0] * relu_derivative(hidden_layer_1.outputs[0]) 

print("Gradient with respect to weight 1 of neuron 1:", dMSE_dW11)
print("Gradient with respect to bias of neuron 1:", dMSE_dB1)

In [ ]:
# Compute parameters of second and third neuron in the same way
dMSE_dW21 = output_error_signal * final_layer.neurons[0].weights[1] * relu_derivative(hidden_layer_1.outputs[1]) * input_layer.forward()[0]
dMSE_dB2 = output_error_signal * final_layer.neurons[0].weights[1] * relu_derivative(hidden_layer_1.outputs[1])

# neuron 3
dMSE_dW31 = output_error_signal * final_layer.neurons[0].weights[2] * relu_derivative(hidden_layer_1.outputs[2]) * input_layer.forward()[0]
dMSE_dB3 = output_error_signal * final_layer.neurons[0].weights[2] * relu_derivative(hidden_layer_1.outputs[2])

In [ ]:
# Update weights and biases of the first layer neurons using gradient descent
hidden_layer_1.neurons[0].update(
    loss_gradient=dMSE_dW11, 
    bias_gradient=dMSE_dB1, 
    learning_rate=learning_rate
)
hidden_layer_1.neurons[1].update(
    loss_gradient=dMSE_dW21, 
    bias_gradient=dMSE_dB2, 
    learning_rate=learning_rate
)
hidden_layer_1.neurons[2].update(
    loss_gradient=dMSE_dW31, 
    bias_gradient=dMSE_dB3, 
    learning_rate=learning_rate
)
print("Updated weights and bias for the first layer neurons:")
for i, neuron in enumerate(hidden_layer_1.neurons):
    print(f"Neuron {i+1}:")
    print("Weights:", neuron.weights)
    print("Bias:", neuron.bias)

In [ ]:
# Compute new output of the network with updated weights and biases for all neurons and show how loss landscape looks around the new prediction point
updated_hidden_layer_output = hidden_layer_1.forward()
updated_final_layer_output_2 = final_layer.forward()
updated_loss_value_after_hidden_layer_update = mean_squared_error(output_vector, updated_final_layer_output_2)





In [ ]:
# show the loss landscape around the new prediction point after updating the weights of the first layer neurons
# and compare it with the previous loss landscape after updating the weights of the last layer neuron only
# and also show the loss landscape before any updates for reference

plt.figure(figsize=(5 ,3))
plt.plot(y_pred_around_true, mse_values, label='Loss Landscape Before Updates')
plt.xlabel('Final Layer Output (Predicted Value)')
plt.ylabel('Mean Squared Error')
plt.scatter(final_output, loss_value, color='red', label='Prediction Before Updates')
plt.scatter(final_output_after_one_update, updated_loss_value, color='orange', label='Prediction After Last Layer Update')
plt.scatter(updated_final_layer_output_2, updated_loss_value_after_hidden_layer_update, color='green', label='Prediction After First Layer Update')
plt.legend();


# Building a complete neural network from scratch
So, we have now run through a simple neural network with one hidden layer. We had a forward pass, then calculated the loss from which we arrived at the output error signal. We backpropagated the error signal throughout the network and saw what happens when we update the weights of each layer in the direction of the negative gradient.

Now let us build a network in a more modular way which helps us train it better. For this, we will be using classes. 

Let us start with the basic element, a neuron. We will create a class called "Neuron" which will have the following attributes:
- weights: a list of weights for each input
- bias: a single bias term
- activation_function: a function which takes the input and produces the output of the neuron

The Neuron class will also have the following methods:
- forward: which takes the input and produces the output of the neuron. It will use the activation function that is already defined above. 

In [ ]:
class Neuron:
    def __init__(self, weights=None, bias=None, input_vector=None, activation_function_type="relu"):
        self.weights = weights if weights is not None else np.random.normal(0, 1) # initialize weights randomly if not provided
        self.bias = bias if bias is not None else np.random.normal(0, 1)
        self.input_vector = input_vector if input_vector is not None else np.random.normal(0, 1, size=(5,)) # initialize input vector randomly if not provided
        self.activation_function_type = activation_function_type
            
    def forward(self):
        z = np.dot(self.weights, self.input_vector) + self.bias
        output = activation_function(z, function_type=self.activation_function_type)
        return output
    
    def update(self, loss_gradient, bias_gradient, learning_rate=0.01):
        # update weights and bias based on the loss gradient and learning rate 
        self.weights = (self.weights - learning_rate * loss_gradient ).squeeze()
        self.bias = np.array([(self.bias - learning_rate * bias_gradient).squeeze()])

class Layer:
    def __init__(self):
        self.neurons = []
        self.outputs = [] # adds a new attribute to store the outputs of the neurons in the layer
    
    def add_neuron(self, neuron):
        # to add single neuron to the layer
        self.neurons.append(neuron)
    
    def add_neurons(self, neurons):
        # to add multiple neurons to the layer
        self.neurons.extend(neurons)
        
    def forward(self):
        outputs = []
        for neuron in self.neurons:
            output = neuron.forward()
            outputs.append(output)
        
        outputs = np.array(outputs)
        #outputs = outputs.squeeze() 
        self.outputs = outputs
        return outputs
    
    def update(self, loss_gradient, learning_rate=0.01):
        # update each neuron in the layer based on the loss gradient and learning rate
        for neuron in self.neurons:
            neuron.update(loss_gradient, learning_rate)

        

In [ ]:
class NeuralNetwork:
    def __init__(self):
        self.layers = []
    
    def add_layer(self, layer):
        self.layers.append(layer)
    
    def forward(self, input_vector):
        current_output = input_vector
        for layer in self.layers:
            current_output = layer.forward()
        return current_output
    
    def update(self, loss_gradient, learning_rate=0.01):
        # Update the network weights using backpropagation
        for layer in reversed(self.layers):
            layer.update(loss_gradient, learning_rate)
    
    def get_gradient

Next, we will create a class called "Layer" which will consist of multiple neurons. The Layer class will have the following attributes:
- neurons: a list of Neuron objects which are part of the layer. Initially, this list can be empty and we can add neurons to it using the add_neuron method.
The Layer class will also have the following methods:
- forward: which takes the input and produces the output of the layer by passing the input through each neuron in the layer and collecting their outputs.
- add_neuron: which allows us to add a neuron to the layer



Finally, we will create a class called "NeuralNetwork" which will consist of multiple layers. The NeuralNetwork class will have the following attributes:
- layers: a list of Layer objects which are part of the neural network. Initially, this list can be empty and we can add layers to it using the add_layer method. 

The NeuralNetwork class will also have the following methods:
- forward: which takes the input and produces the output of the neural network by passing the input through each layer in the network sequentially.
- add_layer: which allows us to add a layer to the neural network
- loss: which calculates the loss of the neural network given the true values and the predicted values. We will use the MSE loss function that we defined above.
- backpropagate: which performs the backpropagation algorithm to update the weights and biases of the neural network based on the calculated gradients.